# BM25 и dense через RRF

Реранкер исключён из текущего кода и повторного эксперимента: на нашем demo calibration он ухудшил top-1 и увеличил задержку. Исторические измерения сохранены для объяснения решения; это не общий вывод обо всех реранкерах. Holdout не используется.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
REPORT = ROOT / 'reports' / 'hybrid_current.json'
HISTORY = ROOT / 'reports' / 'hybrid_comparison.json'


## Повторный эксперимент

Для настоящего нейросетевого прогона установите `requirements-hybrid.txt` в отдельное окружение и выберите его ядром PyCharm. Модели и веса сохраняются в `work/hf_cache/`. Загрузка занимает время и место. Для просмотра уже измеренных результатов torch не требуется. Чтобы повторить прогон, задайте RUN_EXPERIMENT=True. Не используйте прежний demo holdout для выбора нового метода.


In [ ]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    from scripts.compare_hybrid import run
    result, run_dir = run(ROOT / 'data/demo', ROOT / 'work/hybrid_runs')
    print('Артефакты:', run_dir)
else:
    print('Показываем сохранённый отчёт; новый расчёт не выполняется.')


In [ ]:
historical_report = json.loads(HISTORY.read_text(encoding='utf-8'))
report = result if RUN_EXPERIMENT else json.loads(REPORT.read_text(encoding='utf-8'))
active_methods = ['bm25', 'dense', 'hybrid_rrf']
table = pd.DataFrame([{key: value for key, value in metrics.items() if key != 'languages'} | {'method': method} for method, metrics in report['metrics'].items() if method in active_methods])
display(table)
print('Выбор только по calibration:', report['selected_on_calibration'])
print('Holdout открыт:', report['holdout_opened'])
print('Зафиксированные модели:', report['revisions'])


## Языковые срезы и история решения

Рабочее сравнение содержит BM25, dense и hybrid_rrf. Ни один score не является готовой вероятностью правильного ответа. Старые результаты реранкера ниже показываются только как история отклонённого варианта.


In [ ]:
language_rows = []
for method, metrics in report['metrics'].items():
    if method not in active_methods:
        continue
    for language, values in metrics['languages'].items():
        language_rows.append({'method': method, 'language': language, **values})
display(pd.DataFrame(language_rows))
print('Историческое решение:', historical_report['decision']['reason'])
display(pd.DataFrame(historical_report['reranker_changes']))


## Формула RRF

RRF(d) = сумма по выдачам 1 / (c + rank(d)). Ранги начинаются с 1; отсутствующий кандидат даёт нулевой вклад. В этом эксперименте c=60. Каждая ветка возвращает 4 чанка, их объединение даёт до 8 кандидатов; из объединения выбираются первые 5. Объединение выполняется по chunk_id, а метрики — по уникальным document_id. RRF — один из способов fusion; после надёжной нормализации возможна и взвешенная сумма score.
